In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os
import shutil

# Actual database location found in Drive
DRIVE_DB_PATH = "/content/drive/MyDrive/Demand_Forecasting_Inventory_Optimization/Database/demand_forecast.db"

# Local Colab working copy
LOCAL_DB_PATH = "/content/demand_forecast.db"

# Copy database
if os.path.exists(DRIVE_DB_PATH):
    shutil.copy(DRIVE_DB_PATH, LOCAL_DB_PATH)
    print("Database copied to local storage.")
else:
    raise FileNotFoundError("Database not found.")

print("Working Database:")
print(LOCAL_DB_PATH)

Database copied to local storage.
Working Database:
/content/demand_forecast.db


In [6]:
import sqlite3
import pandas as pd
conn = sqlite3.connect("/content/demand_forecast.db")
cursor = conn.cursor()
print("Database connected successfully.")
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    ORDER BY name
    """,
    conn
)
print("\n=== AVAILABLE TABLES ===")
display(tables)

Database connected successfully.

=== AVAILABLE TABLES ===


,name
0,dim_date
1,dim_item
2,dim_model
3,dim_store
4,fact_daily_sales
5,fact_forecast_accuracy
6,fact_inventory_policy
7,fact_service_level_sweep
8,sku_error_summary
9,stg_sales_raw


In [7]:
import pandas as pd
import numpy as np
demand_history = pd.read_sql_query(
    """
    SELECT
        store_key,
        item_key,
        date_key,
        sales_qty
    FROM fact_daily_sales
    ORDER BY store_key, item_key, date_key
    """,
    conn
)
rop_values = pd.read_sql_query(
    """
    SELECT
        store_key,
        item_key,
        naive_reorder_point,
        xgb_reorder_point
    FROM fact_inventory_policy
    """,
    conn
)


print(f"Demand history rows: {len(demand_history):,}")

print(f"ROP reference rows: {len(rop_values)}")
merged_check = demand_history.merge(
    rop_values,
    on=["store_key", "item_key"],
    how="left"
)


n_unmatched = (
    merged_check["naive_reorder_point"]
    .isnull()
    .sum()
)


print(
    f"Demand rows with no matching ROP: {n_unmatched}"
)


assert len(demand_history) == 913000
assert len(rop_values) == 500
assert n_unmatched == 0


print("\nBacktest inputs loaded and validated.")

Demand history rows: 913,000
ROP reference rows: 500
Demand rows with no matching ROP: 0

Backtest inputs loaded and validated.


In [9]:
# Cell 4 (Rebuilt) — Forecast-anchored stockout backtest

# Actual demand for test period
actual_demand = pd.read_sql_query("""
    SELECT
        f.store_key,
        f.item_key,
        f.date_key,
        f.sales_qty AS actual_demand
    FROM fact_daily_sales f
    JOIN dim_date d
        ON f.date_key = d.date_key
    WHERE d.full_date BETWEEN '2017-10-01' AND '2017-12-31'
    ORDER BY f.store_key, f.item_key, f.date_key
""", conn)


# Naive forecast
naive_forecast = pd.read_sql_query("""
    SELECT
        store_key,
        item_key,
        date_key,
        forecast_qty AS naive_forecast
    FROM fact_forecast_accuracy
    WHERE model_key = 1
""", conn)


# XGBoost forecast
xgb_forecast = pd.read_sql_query("""
    SELECT
        store_key,
        item_key,
        date_key,
        forecast_qty AS xgb_forecast
    FROM fact_forecast_accuracy
    WHERE model_key = 4
""", conn)


# Merge actual + forecasts
bt = actual_demand.merge(
    naive_forecast,
    on=['store_key','item_key','date_key'],
    how='inner'
)

bt = bt.merge(
    xgb_forecast,
    on=['store_key','item_key','date_key'],
    how='inner'
)


bt = bt.sort_values(
    ['store_key','item_key','date_key']
)


print(
    f"Test-period rows with both forecasts present: {len(bt):,}"
)


assert len(bt) == 46000



# Rolling 7-day sums
grp = bt.groupby(
    ['store_key','item_key']
)

bt['actual_7d'] = (
    grp['actual_demand']
    .transform(lambda x: x.rolling(7).sum())
)

bt['naive_forecast_7d'] = (
    grp['naive_forecast']
    .transform(lambda x: x.rolling(7).sum())
)

bt['xgb_forecast_7d'] = (
    grp['xgb_forecast']
    .transform(lambda x: x.rolling(7).sum())
)


bt = bt.dropna(
    subset=[
        'actual_7d',
        'naive_forecast_7d',
        'xgb_forecast_7d'
    ]
)



# Add safety stock
rop_values = pd.read_sql_query(
    """
    SELECT
        store_key,
        item_key,
        naive_safety_stock,
        xgb_safety_stock
    FROM fact_inventory_policy
    """,
    conn
)


bt = bt.merge(
    rop_values,
    on=['store_key','item_key'],
    how='left'
)



# Policy-specific thresholds
bt['naive_threshold'] = (
    bt['naive_forecast_7d']
    +
    bt['naive_safety_stock']
)

bt['xgb_threshold'] = (
    bt['xgb_forecast_7d']
    +
    bt['xgb_safety_stock']
)



# Stockout events
bt['naive_stockout'] = (
    bt['actual_7d']
    >
    bt['naive_threshold']
)

bt['xgb_stockout'] = (
    bt['actual_7d']
    >
    bt['xgb_threshold']
)



print(
    f"Valid rolling windows: {len(bt):,}"
)



sku_stockout = (
    bt.groupby(
        ['store_key','item_key']
    )
    .agg(
        naive_stockout_rate=('naive_stockout','mean'),
        xgb_stockout_rate=('xgb_stockout','mean'),
        n_windows=('naive_stockout','count')
    )
    .reset_index()
)



portfolio_naive_rate = (
    sku_stockout['naive_stockout_rate'].mean()
    * 100
)

portfolio_xgb_rate = (
    sku_stockout['xgb_stockout_rate'].mean()
    * 100
)


print(
    f"\nSKUs evaluated: {len(sku_stockout)}"
)

print(
    f"Portfolio avg naive stockout rate: {portfolio_naive_rate:.2f}%"
)

print(
    f"Portfolio avg XGBoost stockout rate: {portfolio_xgb_rate:.2f}%"
)


assert len(sku_stockout) == 500


print(
    "\nForecast-anchored backtest completed."
)

Test-period rows with both forecasts present: 46,000
Valid rolling windows: 43,000

SKUs evaluated: 500
Portfolio avg naive stockout rate: 4.12%
Portfolio avg XGBoost stockout rate: 5.55%

Forecast-anchored backtest completed.


In [10]:
bias_check = pd.read_sql_query("""
    SELECT
        fa.model_key,
        m.model_name,
        fa.actual_qty,
        fa.forecast_qty,
        d.month
    FROM fact_forecast_accuracy fa
    JOIN dim_model m
        ON fa.model_key = m.model_key
    JOIN dim_date d
        ON fa.date_key = d.date_key
    WHERE fa.model_key IN (1, 4)
""", conn)
bias_check['error'] = (
    bias_check['actual_qty']
    -
    bias_check['forecast_qty']
)


print("=== MEAN BIAS (actual - forecast), full test period ===")

print(
    bias_check
    .groupby('model_name')['error']
    .agg(['mean', 'std', 'count'])
)


print("\n=== MEAN BIAS BY MONTH (Oct/Nov/Dec) ===")

print(
    bias_check
    .groupby(['model_name', 'month'])['error']
    .mean()
    .unstack(level=0)
)

=== MEAN BIAS (actual - forecast), full test period ===
                mean        std  count
model_name                            
Naive      -1.334804  11.976630  46000
XGBoost     0.472769   7.954044  46000

=== MEAN BIAS BY MONTH (Oct/Nov/Dec) ===
model_name     Naive   XGBoost
month                         
10         -0.982129  0.149426
11          1.042333  0.564256
12         -3.987935  0.707575


In [12]:
import shutil

DRIVE_DB_PATH = "/content/drive/MyDrive/Demand_Forecasting_Inventory_Optimization/Database/demand_forecast.db"
LOCAL_DB_PATH = "/content/demand_forecast.db"

shutil.copy(
    LOCAL_DB_PATH,
    DRIVE_DB_PATH
)

print("Database copied back to Google Drive.")

Database copied back to Google Drive.


In [13]:
print("\nHonest empirical result persisted.")
print("Naive stockout risk: 4.12%")
print("XGBoost stockout risk: 5.55%")
print("Bias diagnostic retained in notebook history.")


Honest empirical result persisted.
Naive stockout risk: 4.12%
XGBoost stockout risk: 5.55%
Bias diagnostic retained in notebook history.


In [14]:
cursor.execute("DROP TABLE IF EXISTS dim_policy_type")
cursor.execute("""
CREATE TABLE dim_policy_type (
    policy_key INTEGER PRIMARY KEY,
    policy_name TEXT NOT NULL
)
""")

conn.commit()


policy_rows = [
    (1, 'Naive-based'),
    (2, 'ML-based (XGBoost)')
]


cursor.executemany(
    """
    INSERT INTO dim_policy_type
    (policy_key, policy_name)
    VALUES (?, ?)
    """,
    policy_rows
)

conn.commit()


result = pd.read_sql_query(
    "SELECT * FROM dim_policy_type",
    conn
)

print(result)

assert len(result) == 2

print("\ndim_policy_type created and validated.")

   policy_key         policy_name
0           1         Naive-based
1           2  ML-based (XGBoost)

dim_policy_type created and validated.


In [17]:
wide = pd.read_sql_query(
    "SELECT * FROM fact_inventory_policy",
    conn
)

common_cols = [
    'store_key',
    'item_key',
    'avg_daily_demand',
    'annual_demand',
    'unit_price',
    'eoq',
    'best_model_name',
    'service_level',
    'z_value'
]

naive_long = wide[common_cols].copy()

naive_long['policy_key'] = 1
naive_long['forecast_error_sigma'] = wide['naive_sigma']
naive_long['safety_stock'] = wide['naive_safety_stock']
naive_long['reorder_point'] = wide['naive_reorder_point']
naive_long['avg_inventory'] = wide['naive_avg_inventory']
naive_long['holding_cost'] = wide['naive_holding_cost']
xgb_long = wide[common_cols].copy()

xgb_long['policy_key'] = 2
xgb_long['forecast_error_sigma'] = wide['best_sigma']
xgb_long['safety_stock'] = wide['xgb_safety_stock']
xgb_long['reorder_point'] = wide['xgb_reorder_point']
xgb_long['avg_inventory'] = wide['xgb_avg_inventory']
xgb_long['holding_cost'] = wide['xgb_holding_cost']
inventory_policy_long = pd.concat(
    [naive_long, xgb_long],
    ignore_index=True
)

inventory_policy_long = (
    inventory_policy_long
    .sort_values(
        ['store_key','item_key','policy_key']
    )
    .reset_index(drop=True)
)


print(
    f"Long-format rows: {len(inventory_policy_long)}"
)

print(
    inventory_policy_long.head(4)
)
wide_total_naive_ss = wide['naive_safety_stock'].sum()

long_total_naive_ss = (
    inventory_policy_long[
        inventory_policy_long.policy_key == 1
    ]['safety_stock']
    .sum()
)


print(
    f"\nCross-check -- naive safety stock total:"
    f" wide={wide_total_naive_ss:.1f},"
    f" long={long_total_naive_ss:.1f}"
)


assert len(inventory_policy_long) == 1000

assert abs(
    wide_total_naive_ss - long_total_naive_ss
) < 0.01
cursor.execute(
    "DROP TABLE IF EXISTS fact_inventory_policy_long"
)


cursor.execute("""
CREATE TABLE fact_inventory_policy_long (
    store_key INTEGER NOT NULL,
    item_key INTEGER NOT NULL,
    policy_key INTEGER NOT NULL,
    avg_daily_demand REAL,
    annual_demand REAL,
    unit_price REAL,
    eoq REAL,
    best_model_name TEXT,
    service_level REAL,
    z_value REAL,
    forecast_error_sigma REAL,
    safety_stock REAL,
    reorder_point REAL,
    avg_inventory REAL,
    holding_cost REAL,
    PRIMARY KEY (store_key,item_key,policy_key),
    FOREIGN KEY (store_key) REFERENCES dim_store(store_key),
    FOREIGN KEY (item_key) REFERENCES dim_item(item_key),
    FOREIGN KEY (policy_key) REFERENCES dim_policy_type(policy_key)
)
""")

conn.commit()


cols = [
    'store_key',
    'item_key',
    'policy_key',
    'avg_daily_demand',
    'annual_demand',
    'unit_price',
    'eoq',
    'best_model_name',
    'service_level',
    'z_value',
    'forecast_error_sigma',
    'safety_stock',
    'reorder_point',
    'avg_inventory',
    'holding_cost'
]


cursor.executemany(
    f"""
    INSERT INTO fact_inventory_policy_long
    ({','.join(cols)})
    VALUES ({','.join(['?']*len(cols))})
    """,
    inventory_policy_long[cols]
    .itertuples(index=False, name=None)
)

conn.commit()


n_check = pd.read_sql_query(
    """
    SELECT COUNT(*) AS n
    FROM fact_inventory_policy_long
    """,
    conn
)['n'][0]


assert n_check == 1000


print(
    f"\nfact_inventory_policy_long: {n_check} rows in database, validated."
)

Long-format rows: 1000
   store_key  item_key  avg_daily_demand  annual_demand  unit_price  \
0          1         1         22.172603         8093.0     2297.68   
1          1         1         22.172603         8093.0     2297.68   
2          1         2         59.098630        21571.0      235.35   
3          1         2         59.098630        21571.0      235.35   

          eoq best_model_name  service_level  z_value  policy_key  \
0  162.532655         XGBoost           0.95    1.645           1   
1  162.532655         XGBoost           0.95    1.645           2   
2  829.103350         XGBoost           0.95    1.645           1   
3  829.103350         XGBoost           0.95    1.645           2   

   forecast_error_sigma  safety_stock  reorder_point  avg_inventory  \
0              6.357586     27.669872     182.878091     108.936199   
1              4.755867     20.698772     175.906991     101.965100   
2             10.559571     45.958008     459.648419     460.5

In [18]:
# Cell 8 — Unpivot fact_service_level_sweep (wide → long)

wide_sweep = pd.read_sql_query(
    "SELECT * FROM fact_service_level_sweep",
    conn
)


common_cols = [
    'store_key',
    'item_key',
    'service_level',
    'z_value'
]


# Naive policy
naive_sweep_long = wide_sweep[common_cols].copy()

naive_sweep_long['policy_key'] = 1
naive_sweep_long['safety_stock'] = wide_sweep['naive_safety_stock']
naive_sweep_long['reorder_point'] = wide_sweep['naive_reorder_point']
naive_sweep_long['avg_inventory'] = wide_sweep['naive_avg_inventory']
naive_sweep_long['holding_cost'] = wide_sweep['naive_holding_cost']


# XGBoost policy
xgb_sweep_long = wide_sweep[common_cols].copy()

xgb_sweep_long['policy_key'] = 2
xgb_sweep_long['safety_stock'] = wide_sweep['xgb_safety_stock']
xgb_sweep_long['reorder_point'] = wide_sweep['xgb_reorder_point']
xgb_sweep_long['avg_inventory'] = wide_sweep['xgb_avg_inventory']
xgb_sweep_long['holding_cost'] = wide_sweep['xgb_holding_cost']


# Combine
sweep_long = pd.concat(
    [naive_sweep_long, xgb_sweep_long],
    ignore_index=True
)


sweep_long = (
    sweep_long
    .sort_values(
        ['store_key','item_key','service_level','policy_key']
    )
    .reset_index(drop=True)
)


print(
    f"Long-format rows: {len(sweep_long)}"
)

print(
    sweep_long.head(4)
)


# 95% slice validation
wide_95_naive_ss = (
    wide_sweep.loc[
        wide_sweep.service_level == 0.95,
        'naive_safety_stock'
    ]
    .sum()
)


long_95_naive_ss = (
    sweep_long.loc[
        (sweep_long.service_level == 0.95)
        &
        (sweep_long.policy_key == 1),
        'safety_stock'
    ]
    .sum()
)


print(
    f"\n95% slice cross-check -- naive safety stock:"
    f" wide={wide_95_naive_ss:.1f},"
    f" long={long_95_naive_ss:.1f}"
)


# Monotonicity check
sweep_check = (
    sweep_long
    .groupby(
        ['service_level','policy_key']
    )['safety_stock']
    .mean()
    .unstack()
)


is_monotonic = (
    sweep_check[1].is_monotonic_increasing
    and
    sweep_check[2].is_monotonic_increasing
)


assert len(sweep_long) == 10000

assert abs(
    wide_95_naive_ss - long_95_naive_ss
) < 0.01

assert is_monotonic


# Create table
cursor.execute(
    "DROP TABLE IF EXISTS fact_service_level_sweep_long"
)


cursor.execute("""
CREATE TABLE fact_service_level_sweep_long (
    store_key INTEGER NOT NULL,
    item_key INTEGER NOT NULL,
    policy_key INTEGER NOT NULL,
    service_level REAL NOT NULL,
    z_value REAL,
    safety_stock REAL,
    reorder_point REAL,
    avg_inventory REAL,
    holding_cost REAL,
    PRIMARY KEY (
        store_key,
        item_key,
        policy_key,
        service_level
    ),
    FOREIGN KEY (store_key) REFERENCES dim_store(store_key),
    FOREIGN KEY (item_key) REFERENCES dim_item(item_key),
    FOREIGN KEY (policy_key) REFERENCES dim_policy_type(policy_key)
)
""")

conn.commit()


cols = [
    'store_key',
    'item_key',
    'policy_key',
    'service_level',
    'z_value',
    'safety_stock',
    'reorder_point',
    'avg_inventory',
    'holding_cost'
]


cursor.executemany(
    f"""
    INSERT INTO fact_service_level_sweep_long
    ({','.join(cols)})
    VALUES ({','.join(['?']*len(cols))})
    """,
    sweep_long[cols].itertuples(index=False, name=None)
)

conn.commit()


n_check = pd.read_sql_query(
    """
    SELECT COUNT(*) AS n
    FROM fact_service_level_sweep_long
    """,
    conn
)['n'][0]


assert n_check == 10000


print(
    f"\nfact_service_level_sweep_long: {n_check} rows in database, validated."
)

Long-format rows: 10000
   store_key  item_key  service_level   z_value  policy_key  safety_stock  \
0          1         1           0.90  1.281552           1     21.556454   
1          1         1           0.90  1.281552           2     16.125559   
2          1         1           0.91  1.340755           1     22.552292   
3          1         1           0.91  1.340755           2     16.870506   

   reorder_point  avg_inventory  holding_cost  
0     176.764674     102.822782  47250.769944  
1     171.333778      97.391886  44755.077805  
2     177.760511     103.818619  47708.393039  
3     172.078726      98.136834  45097.408129  

95% slice cross-check -- naive safety stock: wide=25114.8, long=25114.8

fact_service_level_sweep_long: 10000 rows in database, validated.


In [19]:
import os
exports_dir = os.path.join(PROJECT_ROOT, "exports")
os.makedirs(
    exports_dir,
    exist_ok=True
)
tables_to_export = [
    "dim_date",
    "dim_store",
    "dim_item",
    "dim_model",
    "dim_policy_type",
    "fact_daily_sales",
    "fact_forecast_accuracy",
    "fact_inventory_policy_long",
    "fact_service_level_sweep_long",
    "sku_error_summary",
    "stockout_risk_summary"
]


export_log = []


for table in tables_to_export:

    df = pd.read_sql_query(
        f"SELECT * FROM {table}",
        conn
    )

    filepath = os.path.join(
        exports_dir,
        f"{table}.csv"
    )

    df.to_csv(
        filepath,
        index=False
    )
    reread = pd.read_csv(filepath)

    n_source = len(df)
    n_file = len(reread)

    match = n_source == n_file

    export_log.append(
        (
            table,
            n_source,
            n_file,
            match
        )
    )


    print(
        f"{table:35s} "
        f"source={n_source:>7,}  "
        f"file={n_file:>7,}  "
        f"match={match}"
    )


all_match = all(
    row[3]
    for row in export_log
)


print(
    f"\nAll {len(tables_to_export)} exports match their source table row counts: {all_match}"
)


assert all_match, (
    "At least one export row count does not match source table"
)


print(
    f"\nFiles written to: {exports_dir}"
)

dim_date                            source=  1,826  file=  1,826  match=True
dim_store                           source=     10  file=     10  match=True
dim_item                            source=     50  file=     50  match=True
dim_model                           source=      4  file=      4  match=True
dim_policy_type                     source=      2  file=      2  match=True
fact_daily_sales                    source=913,000  file=913,000  match=True
fact_forecast_accuracy              source=184,000  file=184,000  match=True
fact_inventory_policy_long          source=  1,000  file=  1,000  match=True
fact_service_level_sweep_long       source= 10,000  file= 10,000  match=True
sku_error_summary                   source=    500  file=    500  match=True
stockout_risk_summary               source=    500  file=    500  match=True

All 11 exports match their source table row counts: True

Files written to: /content/drive/MyDrive/demand_forecasting_inventory/exports


In [21]:
forecast_acc = pd.read_csv(
    os.path.join(exports_dir, "fact_forecast_accuracy.csv")
)

dim_model = pd.read_csv(
    os.path.join(exports_dir, "dim_model.csv")
)

forecast_acc = forecast_acc.merge(
    dim_model,
    on="model_key"
)

forecast_acc_valid = forecast_acc.dropna(
    subset=["abs_pct_error"]
)

mape_by_model = (
    forecast_acc_valid
    .groupby(
        ["model_key", "model_name"]
    )["abs_pct_error"]
    .mean()
    * 100
)


print("=== FORECASTING (from CSV) ===")


benchmarks = {
    1: ("Naive", 19.9),
    2: ("SARIMA", 25.2),
    3: ("Holt-Winters", 23.4),
    4: ("XGBoost", 13.3)
}


for model_key, (name, bench) in benchmarks.items():

    actual = float(
        mape_by_model.loc[
            model_key
        ]
    )

    print(
        f"{name:15s} "
        f"MAPE = {actual:5.1f}% "
        f"(benchmark: {bench}%) "
        f"match: {abs(actual-bench)<0.5}"
    )

    assert abs(actual - bench) < 0.5


print("\nForecast validation passed.")

=== FORECASTING (from CSV) ===
Naive           MAPE =  19.9% (benchmark: 19.9%) match: True
SARIMA          MAPE =  25.2% (benchmark: 25.2%) match: True
Holt-Winters    MAPE =  23.4% (benchmark: 23.4%) match: True
XGBoost         MAPE =  13.3% (benchmark: 13.3%) match: True

Forecast validation passed.


/tmp/ipykernel_5479/824489506.py:41: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  actual = float(


In [23]:

import pandas as pd
import os


exports_dir = os.path.join(
    PROJECT_ROOT,
    "exports"
)


forecast_acc = pd.read_csv(
    os.path.join(
        exports_dir,
        "fact_forecast_accuracy.csv"
    )
)

dim_model = pd.read_csv(
    os.path.join(
        exports_dir,
        "dim_model.csv"
    )
)


forecast_acc = forecast_acc.merge(
    dim_model,
    on="model_key"
)


forecast_acc_valid = forecast_acc.dropna(
    subset=["abs_pct_error"]
)


mape_by_model = (
    forecast_acc_valid
    .groupby(
        ["model_key", "model_name"]
    )["abs_pct_error"]
    .mean()
    .mul(100)
)


print("=== FORECASTING (from CSV) ===")


benchmarks = {
    1: ("Naive", 19.9),
    2: ("SARIMA", 25.2),
    3: ("Holt-Winters", 23.4),
    4: ("XGBoost", 13.3)
}


for model_key, (name, bench) in benchmarks.items():

    actual = (
        mape_by_model
        .xs(model_key, level="model_key")
        .iloc[0]
    )

    print(
        f"{name:15s} "
        f"MAPE = {actual:5.1f}% "
        f"(benchmark: {bench}%) "
        f"match: {abs(actual-bench)<0.5}"
    )

    assert abs(actual - bench) < 0.5


print("\nForecast validation passed.")

inv_long = pd.read_csv(
    os.path.join(
        exports_dir,
        "fact_inventory_policy_long.csv"
    )
)


inv_95 = inv_long[
    inv_long["service_level"] == 0.95
]


ss_naive = (
    inv_95[
        inv_95["policy_key"] == 1
    ]["safety_stock"]
    .sum()
)


ss_xgb = (
    inv_95[
        inv_95["policy_key"] == 2
    ]["safety_stock"]
    .sum()
)


ss_reduction = (
    1 - (ss_xgb / ss_naive)
) * 100



hc_naive = (
    inv_95[
        inv_95["policy_key"] == 1
    ]["holding_cost"]
    .sum()
)

hc_xgb = (
    inv_95[
        inv_95["policy_key"] == 2
    ]["holding_cost"]
    .sum()
)


hc_reduction = (
    1 - (hc_xgb / hc_naive)
) * 100


annual_saving = hc_naive - hc_xgb

print("\n=== INVENTORY (from CSV) ===")

print(
    f"Safety Stock Reduction: {ss_reduction:.1f}% "
    "(benchmark: 33.3%)"
)

print(
    f"Total Holding Cost Reduction: {hc_reduction:.1f}% "
    "(benchmark: 5.3%)"
)

print(
    f"Annual Saving: ₹{annual_saving:,.0f}"
)

assert abs(ss_reduction - 33.3) < 0.5
assert abs(hc_reduction - 5.3) < 0.5

stockout = pd.read_csv(
    os.path.join(
        exports_dir,
        "stockout_risk_summary.csv"
    )
)
naive_stockout = (
    stockout["naive_stockout_rate"]
    .mean()
    * 100
)
xgb_stockout = (
    stockout["xgb_stockout_rate"]
    .mean()
    * 100
)
print("\n=== STOCKOUT RISK (from CSV) ===")
print(
    f"Naive: {naive_stockout:.2f}% "
    "(benchmark: 4.12%)"
)
print(
    f"XGBoost: {xgb_stockout:.2f}% "
    "(benchmark: 5.55%)"
)
assert abs(naive_stockout - 4.12) < 0.1
assert abs(xgb_stockout - 5.55) < 0.1
print(
    "ALL RESUME-FACING METRICS VALIDATED FROM EXPORTED CSV FILES"
)
print(
    "Power BI input files confirmed correct."
)

=== FORECASTING (from CSV) ===
Naive           MAPE =  19.9% (benchmark: 19.9%) match: True
SARIMA          MAPE =  25.2% (benchmark: 25.2%) match: True
Holt-Winters    MAPE =  23.4% (benchmark: 23.4%) match: True
XGBoost         MAPE =  13.3% (benchmark: 13.3%) match: True

Forecast validation passed.

=== INVENTORY (from CSV) ===
Safety Stock Reduction: 33.3% (benchmark: 33.3%)
Total Holding Cost Reduction: 5.3% (benchmark: 5.3%)
Annual Saving: ₹728,946

=== STOCKOUT RISK (from CSV) ===
Naive: 4.12% (benchmark: 4.12%)
XGBoost: 5.55% (benchmark: 5.55%)
ALL RESUME-FACING METRICS VALIDATED FROM EXPORTED CSV FILES
Power BI input files confirmed correct.
